# 03 — Evaluate a trained checkpoint

Runs after `01_train_baseline_detector.ipynb` has produced a real checkpoint
(`model_final.pth` or any `model_XXXXXXX.pth` in its `OUTPUT_DIR`). Computes
real detection metrics (mAP, per-class F1) against real DENTEX ground truth.

**Honest scope of what this notebook can and can't do yet**: the detection
metrics section (parts 1-3) is fully real and was verified end-to-end during
development (against an *untrained* checkpoint, which correctly produces 0
predictions and clean 0.0 scores rather than crashing — see part 3's expected
output). The **confidence/deferral analysis section (part 4) is a template,
not working code** — it needs `src/models/detector.py`'s caries-only wrapper
and a confidence head retrained against the *real* trained detector's
features (notebook 02 only validates the confidence head's task against a
stand-in trunk, not the real backbone) — neither exists yet as of this
writing. Part 4 shows the shape of what to build with placeholder/synthetic
data, matching the pattern already used in `src/eval/metrics.py`'s own
`__main__` block.

**A real, non-obvious bug this notebook works around**: `DiffusionDet`'s
`forward()` defaults to `k=0` at inference time, which only returns
`pred_classes_1` (quadrant-level). For the diagnosis/caries predictions this
project actually cares about, you need `model(batch, k=2)`, which returns
`pred_classes_3` (diagnosis) alongside `pred_classes_1`/`pred_classes_2`.
Confirmed by inspecting the output `Instances` fields directly — `k=0`
silently gives you the wrong task's predictions instead of erroring.

## 1. Setup (same as 00/01 — see those for the full explanation)

In [ ]:
!git clone https://github.com/christopherh-88/Carries-Confidence.git
%cd Carries-Confidence
!pip install -q -r requirements-core.txt
!bash scripts/clone_baseline.sh
!pip install -q ninja
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q timm scipy Pillow

import warnings, sys, os
warnings.filterwarnings("ignore")
import pycocotools.mask, pycocotools.coco, pycocotools.cocoeval
import detectron2
from detectron2 import _C
sys.path.insert(0, "external/HierarchicalDet")
from hierarchialdet.config import add_diffusiondet_config
import torch
print("CUDA available:", torch.cuda.is_available())

## 2. Load the dataset and a trained checkpoint

Point `CHECKPOINT_PATH` at the checkpoint from notebook 01's `OUTPUT_DIR`
(attach that notebook's output as a data source, same pattern as the
multi-session workflow described in 01).

In [ ]:
sys.path.insert(0, ".")
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2
from detectron2.data import DatasetCatalog
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer

DATA_ROOT = "/kaggle/input/dentex/DENTEX/training_data/quadrant-enumeration-disease"
CHECKPOINT_PATH = "/kaggle/input/<your-01-output-dataset>/checkpoints/model_final.pth"  # <-- set this

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
val_dicts = DatasetCatalog.get("custom_validation_class")
print("val images:", len(val_dicts))

cfg = get_cfg()
add_diffusiondet_config(cfg)
cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
cfg.MODEL.WEIGHTS = CHECKPOINT_PATH
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(cfg)
DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
model.eval()
print("checkpoint loaded, device:", cfg.MODEL.DEVICE)

## 3. Run inference and compute real detection metrics

`k=2` is required here — see the note at the top. Expected sanity check on
an *untrained* checkpoint: 0 predictions, `coco_map` returns clean
`{mAP: 0.0, mAP50: 0.0, mAP75: 0.0}` (confirmed — `coco_map` handles the
zero-predictions case explicitly, see `src/eval/metrics.py`), and
`per_class_f1` reports the real `n_gt` counts per category with 0
precision/recall. If you see nonzero predictions and non-flat metrics here on
your trained checkpoint, that's the real signal training worked.

In [ ]:
import cv2
import numpy as np
from src.eval.metrics import coco_map, per_class_f1

def eval_mapper(d, target_size=800):
    img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
    h0, w0 = img.shape[:2]
    img_r = cv2.resize(img, (target_size, target_size))
    return ({"image": torch.as_tensor(img_r.transpose(2, 0, 1).astype(np.float32)),
             "height": target_size, "width": target_size},
            (w0 / target_size, h0 / target_size))

predictions = []
with torch.no_grad():
    for d in val_dicts:
        inp, (sx, sy) = eval_mapper(d)
        out = model([inp], k=2)[0]["instances"]  # k=2 -> pred_classes_3 (diagnosis), not pred_classes
        boxes = out.pred_boxes.tensor.cpu().numpy()
        scores = out.scores.cpu().numpy()
        classes = out.pred_classes_3.cpu().numpy()
        for box, score, cls in zip(boxes, scores, classes):
            x1, y1, x2, y2 = box
            predictions.append({
                "image_id": d["image_id"], "category_id": int(cls),
                "bbox": [float(x1*sx), float(y1*sy), float((x2-x1)*sx), float((y2-y1)*sy)],
                "score": float(score),
            })
print("n predictions:", len(predictions))

ground_truth = {
    "images": [{"id": d["image_id"], "file_name": d["file_name"]} for d in val_dicts],
    "annotations": [
        {"id": i, "image_id": d["image_id"], "category_id": a["category_id_3"],
         "bbox": a["bbox"], "iscrowd": a.get("iscrowd", 0), "area": a["bbox"][2] * a["bbox"][3]}
        for i, (d, a) in enumerate((d, a) for d in val_dicts for a in d["annotations"])
    ],
    "categories": [{"id": i, "name": n} for i, n in
                   enumerate(["Impacted", "Caries", "Periapical Lesion", "Deep Caries"])],
}
print("mAP:", coco_map(predictions, ground_truth))
print("per-class F1:", per_class_f1(predictions, ground_truth))

## 4. Confidence/deferral analysis -- TEMPLATE, not working code yet

This needs two things that don't exist as of this writing:
- `src/models/detector.py`'s caries-only wrapper (still a stub — see
  TASKS.md Phase 3)
- A confidence head retrained against the *real* trained detector's FPN p5
  features (notebook 02 validates the architecture against a stand-in trunk
  only, explicitly not the real backbone — see
  `docs/phase3_confidence_head_training.md`)

Once both exist, replace the synthetic `correct`/`confidence` arrays below
with real per-image values: `correct[i]` = whether image i's predictions
matched ground truth well enough (e.g. all GT boxes matched at IoU>=0.5 — a
real definition needs to be chosen, this is a design decision, not specified
yet), `confidence[i]` = the trained confidence head's usability score on that
image. Everything downstream (`ablation_table`, `sweep_decision_thresholds`,
`expected_calibration_error`, the plots) is real, tested code — only the two
input arrays are placeholders here.

In [ ]:
import numpy as np
from src.eval.metrics import (
    area_under_rc, safe_deferral_rate, ablation_table,
    sweep_decision_thresholds, expected_calibration_error,
)
from src.eval.plots import plot_risk_coverage, plot_reliability_diagram

# PLACEHOLDER -- replace with real (correct, confidence) once the confidence
# head is retrained against the real detector (see the note above).
rng = np.random.default_rng(0)
n = len(val_dicts)
correct = rng.random(n) < 0.8          # placeholder
confidence = rng.random(n)             # placeholder

print("AURC (placeholder data):", area_under_rc(correct, confidence))
print("safe deferral rate @ 0.95 (placeholder data):", safe_deferral_rate(correct, confidence, 0.95))
print("ECE (placeholder data):", expected_calibration_error(correct, confidence))

plot_risk_coverage(correct, confidence, "risk_coverage_PLACEHOLDER.png",
                    title="PLACEHOLDER DATA -- replace once confidence head is retrained")
plot_reliability_diagram(correct, confidence, "reliability_PLACEHOLDER.png",
                          title="PLACEHOLDER DATA -- replace once confidence head is retrained")
print("wrote placeholder plots -- these are NOT real results, do not put them in the paper")